# 03 — Sales Orders

Generates sales orders and lines.
Runs in `overwrite` mode (initial load) or `append` mode (incremental).
Requires master data tables to exist.

In [ ]:
%run ./00_helpers

In [ ]:
RANDOM_SEED        = 42
SCHEMA_NAME        = "rockline"
START_DATE         = "2022-01-01"
END_DATE           = ""
DAILY_ORDER_TARGET = 45
MAX_ORDER_LINES    = 12
TRADE_DISCOUNT_MAX_PCT = 15.0
VAT_RATE           = 0.20
WRITE_MODE         = "overwrite"
ORDER_ID_OFFSET    = 0

In [ ]:
import random
from datetime import datetime, date, timedelta
from decimal import Decimal

rng  = random.Random(RANDOM_SEED)
_NOW = datetime.utcnow()

start_dt = date.fromisoformat(START_DATE)
end_dt   = date.fromisoformat(END_DATE) if END_DATE else _NOW.date() - timedelta(days=1)
print(f"Generating sales: {start_dt} to {end_dt}")

## Load Reference Data

In [ ]:
customers_pd = spark.table(f"{SCHEMA_NAME}.dim_customer").toPandas()
products_pd  = spark.table(f"{SCHEMA_NAME}.dim_product").toPandas()
branches_pd  = spark.table(f"{SCHEMA_NAME}.dim_branch").toPandas()
employees_pd = spark.table(f"{SCHEMA_NAME}.dim_employee").toPandas()

# Build lookup structures
products_list = products_pd.to_dict("records")
branch_ids    = list(customers_pd["preferred_branch_id"].dropna().astype(int).unique())
trade_custs   = customers_pd[customers_pd["customer_type"] == "trade"].to_dict("records")
priv_custs    = customers_pd[customers_pd["customer_type"] == "private"].to_dict("records")

# Employees who can take orders, indexed by branch
sales_roles = {"Trade Counter Sales Advisor", "Account Manager", "Branch Manager",
               "Assistant Branch Manager"}
sales_emps_by_branch = {}
for _, e in employees_pd.iterrows():
    if e["job_title"] in sales_roles:
        sales_emps_by_branch.setdefault(int(e["branch_id"]), []).append(int(e["employee_id"]))

# Category weights per customer type (trade vs private)
# trade: heavy on CC/ST/MA/TM; private: heavy on FF/TM/MA
_TRADE_CAT_WEIGHTS   = {1:0.20, 2:0.18, 3:0.15, 4:0.10, 5:0.18, 6:0.08, 7:0.06, 8:0.05}
_PRIVATE_CAT_WEIGHTS = {1:0.05, 2:0.05, 3:0.22, 4:0.10, 5:0.20, 6:0.10, 7:0.08, 8:0.20}
_QTY_RANGES = {
    1: (1, 50),   2: (1, 100), 3: (1, 50),  4: (5, 200),
    5: (10, 500), 6: (1, 20),  7: (5, 100), 8: (1, 10),
}

def pick_products(is_trade, n, rng):
    wts = _TRADE_CAT_WEIGHTS if is_trade else _PRIVATE_CAT_WEIGHTS
    weights = [wts.get(int(p["category_id"]), 0.05) for p in products_list]
    return rng.choices(products_list, weights=weights, k=n)

print(f"Loaded {len(customers_pd)} customers, {len(products_pd)} products, {len(employees_pd)} employees")

## Build Date Spine & Generate Orders

In [ ]:
order_rows = []
line_rows  = []
oid  = ORDER_ID_OFFSET + 1
lid  = ORDER_ID_OFFSET * MAX_ORDER_LINES + 1
days = date_spine(start_dt, end_dt)

for day in days:
    # Poisson-distributed order count with seasonal weighting
    lam         = DAILY_ORDER_TARGET * apply_seasonal_weight(day)
    n_orders    = poisson_sample(lam, rng)
    cutoff_date = end_dt - timedelta(days=5)

    for _ in range(n_orders):
        # Customer selection: 75% trade by volume
        is_trade = rng.random() < 0.75
        custs    = trade_custs if (is_trade and trade_custs) else priv_custs
        cust     = rng.choice(custs)
        cid      = int(cust["customer_id"])
        branch_id = int(cust["preferred_branch_id"]) if rng.random() < 0.70 else rng.choice(branch_ids)

        # Employee
        emps = sales_emps_by_branch.get(branch_id, [1])
        emp_id = rng.choice(emps)

        # Order timing
        order_hour   = rng.randint(7, 16)
        order_minute = rng.randint(0, 59)
        order_ts     = datetime(day.year, day.month, day.day, order_hour, order_minute)

        # Fulfilment
        if is_trade:
            ftype = rng.choices(["collection","delivery","site_delivery"], weights=[0.55,0.30,0.15])[0]
        else:
            ftype = rng.choices(["collection","delivery"], weights=[0.60,0.40])[0]

        del_addr = del_town = del_post = None
        if ftype in ("delivery", "site_delivery"):
            from faker import Faker
            _f = Faker("en_GB")
            del_addr = _f.street_address()
            del_town = _f.city()
            del_post = _f.postcode()

        # Requested delivery
        req_del = business_day_offset(day, rng.randint(1, 3)) if ftype != "collection" else None

        # Order status & dates based on age
        if day < cutoff_date:
            # Old enough to be completed
            status_roll = rng.random()
            if status_roll < 0.02:
                status        = "cancelled"
                desp_date     = None
                del_date      = None
            else:
                status        = "invoiced"
                desp_date     = business_day_offset(day, rng.randint(0, 1))
                del_date      = business_day_offset(desp_date, rng.randint(1, 2)) if ftype != "collection" else desp_date
        else:
            status   = rng.choice(["confirmed", "picking"])
            desp_date = None
            del_date  = None

        # Credit & payment
        is_credit = is_trade and (cust.get("payment_terms_days", 0) or 0) > 0
        pay_due   = None
        pay_recv  = None
        if is_credit and status == "invoiced":
            pay_due  = del_date + timedelta(days=30) if del_date else None
            # ~5% left unpaid for AR demo
            if rng.random() > 0.05 and pay_due:
                pay_recv = pay_due + timedelta(days=rng.randint(-5, 10))

        # Lines
        n_lines  = rng.randint(1, MAX_ORDER_LINES)
        prods    = pick_products(is_trade, n_lines, rng)
        discount = round(rng.uniform(0, TRADE_DISCOUNT_MAX_PCT) / 100, 4) if is_trade else 0.0

        subtotal = Decimal("0.00")
        for ln, prod in enumerate(prods, 1):
            cat_id  = int(prod["category_id"])
            lo, hi  = _QTY_RANGES.get(cat_id, (1, 20))
            qty     = Decimal(str(round(rng.uniform(lo, hi), 3)))
            if is_trade:
                unit_p = Decimal(str(float(prod["trade_price_gbp"]) * (1 - discount)))
            else:
                unit_p = Decimal(str(prod["list_price_gbp"]))
            unit_p  = Decimal(str(round(float(unit_p), 4)))
            net     = Decimal(str(round(float(qty) * float(unit_p), 2)))
            vat     = Decimal(str(round(float(net) * VAT_RATE, 2)))
            cog     = Decimal(str(round(float(qty) * float(prod["standard_cost_gbp"]), 2)))
            subtotal += net
            line_rows.append((
                lid, oid, ln, int(prod["product_id"]),
                qty, unit_p, net, vat, net + vat, cog, _NOW,
            ))
            lid += 1

        vat_total   = Decimal(str(round(float(subtotal) * VAT_RATE, 2)))
        total       = subtotal + vat_total

        order_rows.append((
            oid,
            f"SO-{day.year}-{oid:07d}",
            cid, branch_id, emp_id, order_ts,
            req_del, desp_date, del_date,
            status, ftype,
            del_addr, del_town, del_post,
            subtotal, vat_total, total,
            Decimal(str(round(discount * 100, 2))),
            is_credit,
            pay_due, pay_recv,
            _NOW,
        ))
        oid += 1

print(f"Generated {len(order_rows)} orders, {len(line_rows)} lines")

## Write Tables

In [ ]:
orders_df = to_spark_df(order_rows, SCHEMA_FACT_SALES_ORDER)
orders_df.write.format("delta").mode(WRITE_MODE).saveAsTable(f"{SCHEMA_NAME}.fact_sales_order")
print(f"fact_sales_order: {orders_df.count()} rows")

lines_df = to_spark_df(line_rows, SCHEMA_FACT_SALES_ORDER_LINE)
lines_df.write.format("delta").mode(WRITE_MODE).saveAsTable(f"{SCHEMA_NAME}.fact_sales_order_line")
print(f"fact_sales_order_line: {lines_df.count()} rows")
print(f"Date range covered: {start_dt} to {end_dt}")